In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Point, LineString

# Machine Learning & Feature Selection
from feature_engine.selection import DropConstantFeatures, SmartCorrelatedSelection
from mrmr import mrmr_classif
import umap
import hdbscan

# Geospatial Visualization
from lonboard import Map, ScatterplotLayer, PathLayer
import networkx as nx
from pyvis.network import Network

# ==========================================
# 1. DATA INGESTION & SEGMENTATION
# ==========================================
# Load your primary telemetry dataset
# df = pd.read_csv("telemetry_logs.csv", low_memory=False)

# Mock segmentation for the script to run:
protocols = {
    'DMR': df[df['Protocol'] == 'DMR'].copy() if 'Protocol' in df.columns else pd.DataFrame(),
    'TETRA': df[df['Protocol'] == 'TETRA'].copy() if 'Protocol' in df.columns else pd.DataFrame(),
    'NXDN': df[df['Protocol'] == 'NXDN'].copy() if 'Protocol' in df.columns else pd.DataFrame(),
    'S1C': df[df['Protocol'] == 'S1C'].copy() if 'Protocol' in df.columns else pd.DataFrame()
}

# Load your new secondary dataset containing Lats, Longs, Datetime, and Information
# df_info = pd.read_csv("external_information.csv")
df_info = pd.DataFrame() # Placeholder to prevent errors if file is missing

# ==========================================
# 2. ADAPTIVE TOPOLOGY & FEATURE FUNCTION
# ==========================================
def process_adaptive_topology(df_subset, geo_cols, identifier_cols, label_col='Entity_Type'):
    """Dynamically handles sparse vs plentiful labels for topology and minimal optimal selection."""
    if df_subset.empty:
        return gpd.GeoDataFrame(geometry=[], crs="EPSG:4326"), gpd.GeoDataFrame(geometry=[], crs="EPSG:4326")

    # Step A: Clean Sparse Features
    cleaner = DropConstantFeatures(tol=0.98)
    df_clean = cleaner.fit_transform(df_subset.drop(columns=geo_cols + identifier_cols, errors='ignore'))
    numeric_features = df_clean.select_dtypes(include=[np.number]).fillna(0)
    
    # Step B: Check Label Volume (Adaptive Branching)
    known_labels_count = df_subset[label_col].notnull().sum() if label_col in df_subset.columns else 0
    
    if known_labels_count > 100:  # Plentiful labels
        target_labels = df_subset[label_col]
        df_clustered = df_subset.copy()
    else:  # Sparse or missing labels
        reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, n_components=2, random_state=42)
        embeddings = reducer.fit_transform(numeric_features)
        
        clusterer = hdbscan.HDBSCAN(min_cluster_size=15, min_samples=5)
        df_subset['discovered_network'] = clusterer.fit_predict(embeddings)
        
        df_clustered = df_subset[df_subset['discovered_network'] != -1].copy()
        target_labels = df_clustered['discovered_network'].astype(str)
        numeric_features = numeric_features.loc[df_clustered.index]

    # Step C: Determine Mobile vs Base Station vs Repeater (mRMR Feature Selection)
    try:
        best_features = mrmr_classif(X=numeric_features, y=target_labels, K=10)
        print(f"Top distinguishing features for subset: {best_features}")
    except Exception:
        pass # Skip if insufficient class diversity

    # Step D: Generate Mapping Geometries
    points = [Point(xy) for xy in zip(df_clustered.longitude, df_clustered.latitude)]
    gdf_nodes = gpd.GeoDataFrame(df_clustered, geometry=points, crs="EPSG:4326")
    
    lines = []
    for _, row in df_clustered.iterrows():
        if pd.notnull(row.get('BaseStation_Lon')) and pd.notnull(row.get('BaseStation_Lat')):
            lines.append(LineString([(row.longitude, row.latitude), 
                                     (row['BaseStation_Lon'], row['BaseStation_Lat'])]))
        else:
            lines.append(None)
            
    gdf_edges = gpd.GeoDataFrame(df_clustered, geometry=lines, crs="EPSG:4326").dropna(subset=['geometry'])
    
    return gdf_nodes, gdf_edges

# ==========================================
# 3. EXECUTE FOR EACH PROTOCOL
# ==========================================
dmr_nodes, dmr_edges = process_adaptive_topology(protocols['DMR'], ['latitude', 'longitude'], ['DMR_ID', 'Talkgroup_ID'])
tetra_nodes, tetra_edges = process_adaptive_topology(protocols['TETRA'], ['latitude', 'longitude'], ['MNI', 'SSI'])
nxdn_nodes, nxdn_edges = process_adaptive_topology(protocols['NXDN'], ['latitude', 'longitude'], ['Unit_ID', 'RAN'])
s1c_nodes, s1c_edges = process_adaptive_topology(protocols['S1C'], ['latitude', 'longitude'], ['eNB_ID', 'TAC'])

# ==========================================
# 4. PROCESS THE NEW EXTERNAL "INFORMATION" LAYER
# ==========================================
info_layer = None
if not df_info.empty and 'Longitude' in df_info.columns and 'Latitude' in df_info.columns:
    info_points = [Point(xy) for xy in zip(df_info.Longitude, df_info.Latitude)]
    gdf_info = gpd.GeoDataFrame(df_info, geometry=info_points, crs="EPSG:4326")
    
    # Render as a distinct layer (e.g., bright yellow, larger radius) so it acts as an "Icon"
    info_layer = ScatterplotLayer.from_geopandas(
        gdf_info,
        get_fill_color=[255, 255, 0], # Yellow
        radius_min_pixels=8,          # Noticeably larger than the radios
        pickable=True                 # Exposes 'datetime' and 'information' on hover
    )

# ==========================================
# 5. GENERATE MAP LAYERS & EXPORT TO HTML
# ==========================================
def create_interactive_layers(node_gdf, edge_gdf, node_color, edge_color):
    if node_gdf.empty:
        return None, None
    node_layer = ScatterplotLayer.from_geopandas(node_gdf, get_fill_color=node_color, radius_min_pixels=4, pickable=True)
    edge_layer = PathLayer.from_geopandas(edge_gdf, get_color=edge_color, width_min_pixels=2, pickable=True)
    return node_layer, edge_layer

dmr_node, dmr_edge = create_interactive_layers(dmr_nodes, dmr_edges, [0, 150, 255], [0, 150, 255, 150]) 
tetra_node, tetra_edge = create_interactive_layers(tetra_nodes, tetra_edges, [0, 255, 0], [0, 255, 0, 150]) 
nxdn_node, nxdn_edge = create_interactive_layers(nxdn_nodes, nxdn_edges, [255, 165, 0], [255, 165, 0, 150])
s1c_node, s1c_edge = create_interactive_layers(s1c_nodes, s1c_edges, [255, 0, 0], [255, 0, 0, 150]) 

# Combine all generated layers into a single list
all_layers = [lyr for lyr in [dmr_edge, dmr_node, tetra_edge, tetra_node, nxdn_edge, nxdn_node, s1c_edge, s1c_node, info_layer] if lyr is not None]

# Render the Map and export as a standalone HTML file
network_map = Map(layers=all_layers, show_tooltip=True)
network_map.to_html("interactive_radio_map.html")
print("Map successfully exported to 'interactive_radio_map.html'")

# ==========================================
# 6. GENERATE NETWORK GRAPH & EXPORT TO HTML
# ==========================================
def add_edges_to_graph(G, edges_gdf, source_col, default_target_prefix):
    """Helper to safely build the graph relationships."""
    if edges_gdf.empty:
        return
    for _, row in edges_gdf.iterrows():
        source_node = f"{default_target_prefix}_{str(row.get(source_col, 'Unknown'))}"
        
        # If no explicit destination exists, connect them to the discovered Talkgroup
        if pd.notnull(row.get('BaseStation_ID')):
            target_node = f"Base_{str(row['BaseStation_ID'])}"
        else:
            target_node = f"Talkgroup_{row.get('discovered_network', 'Unknown')}"
            
        G.add_edge(source_node, target_node)

G = nx.Graph()

# Feed each protocol's topological edges into the graph
add_edges_to_graph(G, dmr_edges, 'DMR_ID', 'DMR')
add_edges_to_graph(G, tetra_edges, 'SSI', 'TETRA')
add_edges_to_graph(G, nxdn_edges, 'Unit_ID', 'NXDN')
add_edges_to_graph(G, s1c_edges, 'eNB_UE_S1AP_ID', 'S1C')

# Initialize PyVis network graph
net = Network(height="800px", width="100%", bgcolor="#222222", font_color="white", notebook=True)
net.from_nx(G)

# Add control panel so customers can manipulate the graph physics
net.show_buttons(filter_=['physics'])

# Export the interactive graph
net.write_html("network_topology_graph.html")
print("Network Graph successfully exported to 'network_topology_graph.html'")